# 02. 시계열 분해 + 정상성 검정 + 베이스라인 모델

## 목표
- STL Decomposition으로 Trend / Seasonal / Residual 구조 파악
- ADF / KPSS 검정으로 정상성 확인 → 차분 필요 여부 결정
- 베이스라인 모델 (Naive, Moving Average)로 Day 3~5 성능 비교 기준선 확보

## 대상
- Top 5 상품군: GROCERY I, BEVERAGES, PRODUCE, CLEANING, DAIRY
- 전체 매장 합산 일별 매출 시계열

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
fm._load_fontmanager(try_read_cache=False)
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

from src.evaluation import mape, rmse, mae, evaluate_model, compare_models

print('Setup 완료')

Setup 완료


---
## 1. 데이터 로드

Day 1에서 전처리한 데이터를 로드한다. 분석은 **Train 기간 (2013-01 ~ 2016-12)**에 대해 수행하고,
베이스라인 모델은 **Validation 기간 (2017-01 ~ 2017-06)**에서 평가한다.

In [2]:
full = pd.read_csv('../data/processed/full_data.csv', parse_dates=['date'])
train = pd.read_csv('../data/processed/train.csv', parse_dates=['date'])
val = pd.read_csv('../data/processed/val.csv', parse_dates=['date'])

families = full['family'].unique().tolist()
print(f'전체: {full.shape}, Train: {train.shape}, Val: {val.shape}')
print(f'상품군: {families}')
print(f'Train 기간: {train["date"].min().date()} ~ {train["date"].max().date()}')
print(f'Val 기간:   {val["date"].min().date()} ~ {val["date"].max().date()}')

전체: (8420, 19), Train: (7285, 19), Val: (905, 19)
상품군: ['BEVERAGES', 'CLEANING', 'DAIRY', 'GROCERY I', 'PRODUCE']
Train 기간: 2013-01-01 ~ 2016-12-31
Val 기간:   2017-01-01 ~ 2017-06-30


---
## 2. STL Decomposition

STL(Seasonal and Trend decomposition using Loess)로 시계열을 3가지 성분으로 분해한다:
- **Trend**: 장기적 추세
- **Seasonal**: 반복 패턴 (주간 계절성 period=7)
- **Residual**: Trend + Seasonal로 설명되지 않는 나머지

주간 계절성(`period=7`)을 기본으로 분석한다. 에콰도르 소매 시장에서 요일별 구매 패턴이
가장 뚜렷한 계절성이기 때문이다.

In [3]:
# STL Decomposition: 주간 계절성 (period=7)
fig, axes = plt.subplots(len(families), 4, figsize=(20, 4 * len(families)))

stl_results = {}

for i, family in enumerate(families):
    ts = train[train['family'] == family].set_index('date')['sales'].asfreq('D')
    ts = ts.interpolate(method='linear')  # 결측 보간
    
    stl = STL(ts, period=7, robust=True)
    result = stl.fit()
    stl_results[family] = result
    
    axes[i, 0].plot(ts.index, ts.values, linewidth=0.5)
    axes[i, 0].set_title(f'{family} - 원본', fontsize=10)
    
    axes[i, 1].plot(ts.index, result.trend, color='orange', linewidth=1)
    axes[i, 1].set_title(f'{family} - Trend', fontsize=10)
    
    axes[i, 2].plot(ts.index, result.seasonal, color='green', linewidth=0.5)
    axes[i, 2].set_title(f'{family} - Seasonal', fontsize=10)
    
    axes[i, 3].plot(ts.index, result.resid, color='red', linewidth=0.5, alpha=0.7)
    axes[i, 3].set_title(f'{family} - Residual', fontsize=10)

plt.suptitle('STL Decomposition (period=7, 주간 계절성)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/07_stl_weekly.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: outputs/figures/07_stl_weekly.png')

저장: outputs/figures/07_stl_weekly.png


### 2.1 계절성 강도 (Strength of Seasonality)

계절성 강도는 Residual 대비 Seasonal 성분이 얼마나 큰지를 측정한다:

$$F_S = \max\left(0, 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}\right)$$

- $F_S \approx 1$: 매우 강한 계절성
- $F_S \approx 0$: 계절성 약함

In [4]:
# 계절성 강도 계산
seasonality_strength = []
trend_strength = []

for family in families:
    result = stl_results[family]
    
    # 계절성 강도: 1 - Var(Residual) / Var(Seasonal + Residual)
    var_resid = np.var(result.resid)
    var_seasonal_resid = np.var(result.seasonal + result.resid)
    f_s = max(0, 1 - var_resid / var_seasonal_resid)
    
    # 트렌드 강도: 1 - Var(Residual) / Var(Trend + Residual)
    var_trend_resid = np.var(result.trend + result.resid)
    f_t = max(0, 1 - var_resid / var_trend_resid)
    
    seasonality_strength.append({'family': family, 'F_seasonal': f_s, 'F_trend': f_t})

strength_df = pd.DataFrame(seasonality_strength)
print('계절성/트렌드 강도:')
print(strength_df.to_string(index=False, float_format='{:.4f}'.format))
print()
print('해석:')
for _, row in strength_df.iterrows():
    s_level = '강함' if row['F_seasonal'] > 0.5 else ('보통' if row['F_seasonal'] > 0.3 else '약함')
    t_level = '강함' if row['F_trend'] > 0.5 else ('보통' if row['F_trend'] > 0.3 else '약함')
    print(f"  {row['family']}: 계절성={s_level}({row['F_seasonal']:.3f}), 트렌드={t_level}({row['F_trend']:.3f})")

계절성/트렌드 강도:
   family  F_seasonal  F_trend
BEVERAGES      0.7260   0.8958
 CLEANING      0.6482   0.5502
    DAIRY      0.7716   0.8684
GROCERY I      0.5949   0.6451
  PRODUCE      0.5408   0.9524

해석:
  BEVERAGES: 계절성=강함(0.726), 트렌드=강함(0.896)
  CLEANING: 계절성=강함(0.648), 트렌드=강함(0.550)
  DAIRY: 계절성=강함(0.772), 트렌드=강함(0.868)
  GROCERY I: 계절성=강함(0.595), 트렌드=강함(0.645)
  PRODUCE: 계절성=강함(0.541), 트렌드=강함(0.952)


### 2.2 주간 계절성 패턴 시각화

STL에서 추출한 Seasonal 성분의 1주일(7일) 패턴을 상품군별로 비교한다.
요일별 매출 패턴이 상품군마다 다른지 확인한다.

In [5]:
# 주간 패턴 추출 (7일 반복 구간)
fig, axes = plt.subplots(1, len(families), figsize=(18, 4), sharey=False)
day_labels = ['월', '화', '수', '목', '금', '토', '일']

for i, family in enumerate(families):
    result = stl_results[family]
    seasonal = result.seasonal
    
    # 요일별 평균 계절성
    weekly = seasonal.groupby(seasonal.index.dayofweek).mean()
    
    colors = ['#3498db' if v >= 0 else '#e74c3c' for v in weekly.values]
    axes[i].bar(range(7), weekly.values, color=colors, alpha=0.8)
    axes[i].set_xticks(range(7))
    axes[i].set_xticklabels(day_labels)
    axes[i].set_title(family, fontsize=11, fontweight='bold')
    axes[i].axhline(y=0, color='black', linewidth=0.5)

plt.suptitle('상품군별 주간 계절성 패턴 (STL Seasonal 평균)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/08_weekly_pattern.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: outputs/figures/08_weekly_pattern.png')

저장: outputs/figures/08_weekly_pattern.png


---
## 3. 정상성 검정 (Stationarity Tests)

SARIMA 모델의 차분 차수(d)를 결정하기 위해 정상성을 검정한다.

| 검정 | 귀무가설 | 판단 기준 |
|------|---------|----------|
| **ADF** (Augmented Dickey-Fuller) | 단위근 존재 (비정상) | p < 0.05 → 정상 |
| **KPSS** | 정상 과정 | p > 0.05 → 정상 |

두 검정의 결과가 일치하면 확실하고, 상충하면 추가 분석이 필요하다.

In [6]:
def run_stationarity_tests(series, name=''):
    """ADF + KPSS 검정을 실행하고 결과를 반환한다."""
    series = series.dropna()
    
    # ADF Test
    adf_stat, adf_p, adf_lags, adf_nobs, _, _ = adfuller(series, autolag='AIC')
    
    # KPSS Test
    kpss_stat, kpss_p, kpss_lags, _ = kpss(series, regression='c', nlags='auto')
    
    adf_stationary = adf_p < 0.05
    kpss_stationary = kpss_p > 0.05
    
    return {
        'series': name,
        'adf_stat': adf_stat,
        'adf_p': adf_p,
        'adf_stationary': adf_stationary,
        'kpss_stat': kpss_stat,
        'kpss_p': kpss_p,
        'kpss_stationary': kpss_stationary,
        'both_agree': adf_stationary == kpss_stationary,
        'conclusion': '정상' if (adf_stationary and kpss_stationary) else ('비정상' if (not adf_stationary and not kpss_stationary) else '불확실')
    }

In [7]:
# 원본 시계열 + 1차 차분에 대해 검정
test_results = []

for family in families:
    ts = train[train['family'] == family].set_index('date')['sales'].asfreq('D')
    ts = ts.interpolate(method='linear')
    
    # 원본 시계열
    result_orig = run_stationarity_tests(ts, f'{family} (원본)')
    result_orig['family'] = family
    result_orig['diff'] = 0
    test_results.append(result_orig)
    
    # 1차 차분
    ts_diff = ts.diff().dropna()
    result_diff = run_stationarity_tests(ts_diff, f'{family} (1차 차분)')
    result_diff['family'] = family
    result_diff['diff'] = 1
    test_results.append(result_diff)

test_df = pd.DataFrame(test_results)
print('정상성 검정 결과:')
print('=' * 90)
for family in families:
    sub = test_df[test_df['family'] == family]
    for _, row in sub.iterrows():
        print(f"{row['series']:30s} | ADF p={row['adf_p']:.4f} ({'정상' if row['adf_stationary'] else '비정상'}) | "
              f"KPSS p={row['kpss_p']:.4f} ({'정상' if row['kpss_stationary'] else '비정상'}) | → {row['conclusion']}")
    print('-' * 90)

정상성 검정 결과:
BEVERAGES (원본)                 | ADF p=0.1766 (비정상) | KPSS p=0.0100 (비정상) | → 비정상
BEVERAGES (1차 차분)              | ADF p=0.0000 (정상) | KPSS p=0.1000 (정상) | → 정상
------------------------------------------------------------------------------------------
CLEANING (원본)                  | ADF p=0.0071 (정상) | KPSS p=0.0100 (비정상) | → 불확실
CLEANING (1차 차분)               | ADF p=0.0000 (정상) | KPSS p=0.1000 (정상) | → 정상
------------------------------------------------------------------------------------------
DAIRY (원본)                     | ADF p=0.6942 (비정상) | KPSS p=0.0100 (비정상) | → 비정상
DAIRY (1차 차분)                  | ADF p=0.0000 (정상) | KPSS p=0.1000 (정상) | → 정상
------------------------------------------------------------------------------------------
GROCERY I (원본)                 | ADF p=0.0309 (정상) | KPSS p=0.0100 (비정상) | → 불확실
GROCERY I (1차 차분)              | ADF p=0.0000 (정상) | KPSS p=0.1000 (정상) | → 정상
--------------------------------------------------------------------------

### 3.1 정상성 검정 요약

검정 결과를 상품군별 요약표로 정리한다.

In [8]:
# 요약표: 상품군별 차분 권장
summary_rows = []
for family in families:
    orig = test_df[(test_df['family'] == family) & (test_df['diff'] == 0)].iloc[0]
    diff1 = test_df[(test_df['family'] == family) & (test_df['diff'] == 1)].iloc[0]
    
    if orig['conclusion'] == '정상':
        recommended_d = 0
    elif diff1['conclusion'] == '정상':
        recommended_d = 1
    else:
        recommended_d = 1  # 불확실해도 1차 차분 권장
    
    summary_rows.append({
        '상품군': family,
        'ADF p (원본)': f"{orig['adf_p']:.4f}",
        'KPSS p (원본)': f"{orig['kpss_p']:.4f}",
        '원본 결론': orig['conclusion'],
        'ADF p (차분)': f"{diff1['adf_p']:.4f}",
        'KPSS p (차분)': f"{diff1['kpss_p']:.4f}",
        '차분 결론': diff1['conclusion'],
        '권장 d': recommended_d
    })

summary_df = pd.DataFrame(summary_rows)
print('상품군별 정상성 검정 요약:')
print(summary_df.to_string(index=False))
print()
print('결론: SARIMA 모델링 시 위 권장 d 값을 사용한다.')

상품군별 정상성 검정 요약:
      상품군 ADF p (원본) KPSS p (원본) 원본 결론 ADF p (차분) KPSS p (차분) 차분 결론  권장 d
BEVERAGES     0.1766      0.0100   비정상     0.0000      0.1000    정상     1
 CLEANING     0.0071      0.0100   불확실     0.0000      0.1000    정상     1
    DAIRY     0.6942      0.0100   비정상     0.0000      0.1000    정상     1
GROCERY I     0.0309      0.0100   불확실     0.0000      0.1000    정상     1
  PRODUCE     0.0570      0.0100   비정상     0.0000      0.1000    정상     1

결론: SARIMA 모델링 시 위 권장 d 값을 사용한다.


### 3.2 원본 vs 차분 시각화

In [9]:
fig, axes = plt.subplots(len(families), 2, figsize=(16, 3 * len(families)))

for i, family in enumerate(families):
    ts = train[train['family'] == family].set_index('date')['sales'].asfreq('D')
    ts = ts.interpolate(method='linear')
    ts_diff = ts.diff().dropna()
    
    axes[i, 0].plot(ts.index, ts.values, linewidth=0.5, color='#2c3e50')
    axes[i, 0].set_title(f'{family} - 원본', fontsize=10)
    
    axes[i, 1].plot(ts_diff.index, ts_diff.values, linewidth=0.5, color='#e74c3c')
    axes[i, 1].set_title(f'{family} - 1차 차분', fontsize=10)
    axes[i, 1].axhline(y=0, color='black', linewidth=0.5)

plt.suptitle('원본 시계열 vs 1차 차분', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/09_stationarity.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: outputs/figures/09_stationarity.png')

저장: outputs/figures/09_stationarity.png


---
## 4. 베이스라인 모델

Day 3~5에서 구현할 SARIMA, Prophet, XGBoost의 성능을 평가하기 위한 **기준선(baseline)**을 설정한다.
고급 모델은 최소한 이 베이스라인보다 나아야 의미가 있다.

### 베이스라인 모델 3종
1. **Naive (7일 계절)**: 7일 전 동일 요일 매출을 그대로 사용
2. **Moving Average 7일**: 직전 7일 매출 평균
3. **Moving Average 30일**: 직전 30일 매출 평균

평가 기간: **Validation (2017-01-01 ~ 2017-06-30)**

In [10]:
def naive_seasonal_forecast(full_series, val_dates, lag=7):
    """Seasonal Naive: lag일 전 값을 예측값으로 사용."""
    preds = []
    for date in val_dates:
        past_date = date - pd.Timedelta(days=lag)
        if past_date in full_series.index:
            preds.append(full_series.loc[past_date])
        else:
            preds.append(np.nan)
    return np.array(preds)


def moving_average_forecast(full_series, val_dates, window=7):
    """Moving Average: 직전 window일 평균을 예측값으로 사용."""
    preds = []
    for date in val_dates:
        past_start = date - pd.Timedelta(days=window)
        past_end = date - pd.Timedelta(days=1)
        past_values = full_series.loc[past_start:past_end]
        if len(past_values) >= window // 2:  # 최소 절반 이상 데이터 필요
            preds.append(past_values.mean())
        else:
            preds.append(np.nan)
    return np.array(preds)


print('베이스라인 함수 정의 완료')

베이스라인 함수 정의 완료


In [11]:
# 모든 상품군 x 모든 베이스라인 평가
baseline_results = []

for family in families:
    # Train + Val 연결 시계열 (베이스라인은 직전 값 참조 필요)
    full_ts = full[full['family'] == family].set_index('date')['sales'].asfreq('D')
    full_ts = full_ts.interpolate(method='linear')
    
    val_sub = val[val['family'] == family].sort_values('date')
    val_dates = pd.DatetimeIndex(val_sub['date'])
    y_true = val_sub['sales'].values
    
    # 1. Naive (7일 계절)
    y_naive = naive_seasonal_forecast(full_ts, val_dates, lag=7)
    mask = ~np.isnan(y_naive)
    baseline_results.append(
        evaluate_model(y_true[mask], y_naive[mask], 'Naive_7d', family)
    )
    
    # 2. Moving Average 7일
    y_ma7 = moving_average_forecast(full_ts, val_dates, window=7)
    mask = ~np.isnan(y_ma7)
    baseline_results.append(
        evaluate_model(y_true[mask], y_ma7[mask], 'MA_7d', family)
    )
    
    # 3. Moving Average 30일
    y_ma30 = moving_average_forecast(full_ts, val_dates, window=30)
    mask = ~np.isnan(y_ma30)
    baseline_results.append(
        evaluate_model(y_true[mask], y_ma30[mask], 'MA_30d', family)
    )

baseline_df = compare_models(baseline_results)
print('베이스라인 모델 평가 결과:')
print(baseline_df.to_string(index=False, float_format='{:.2f}'.format))

베이스라인 모델 평가 결과:
   model    family   mape     rmse      mae  train_time_sec  predict_time_sec
Naive_7d BEVERAGES  43.99 44454.39 26633.94            0.00              0.00
  MA_30d BEVERAGES  48.26 50826.18 41546.71            0.00              0.00
   MA_7d BEVERAGES  49.23 51225.19 41144.72            0.00              0.00
  MA_30d  CLEANING 120.03 15995.98 12618.19            0.00              0.00
   MA_7d  CLEANING 129.79 16144.15 12250.73            0.00              0.00
Naive_7d  CLEANING 138.95 17428.79 11497.37            0.00              0.00
  MA_30d     DAIRY  65.78 11989.83  9593.15            0.00              0.00
Naive_7d     DAIRY  66.80 11394.30  6747.73            0.00              0.00
   MA_7d     DAIRY  66.97 12216.17  9531.68            0.00              0.00
   MA_7d GROCERY I  98.14 59451.12 45516.47            0.00              0.00
  MA_30d GROCERY I 105.44 60973.66 47272.32            0.00              0.00
Naive_7d GROCERY I 113.14 61708.00 38498.57     

### 4.1 베이스라인 결과 저장 + 시각화

In [12]:
# CSV 저장
baseline_df.to_csv('../outputs/results/baseline_results.csv', index=False)
print('저장: outputs/results/baseline_results.csv')

저장: outputs/results/baseline_results.csv


In [13]:
# 상품군별 MAPE 비교 차트
fig, ax = plt.subplots(figsize=(12, 6))

pivot = baseline_df.pivot(index='family', columns='model', values='mape')
pivot = pivot[['Naive_7d', 'MA_7d', 'MA_30d']]  # 순서 고정

x = np.arange(len(pivot.index))
width = 0.25

colors = ['#3498db', '#2ecc71', '#e74c3c']
for j, col in enumerate(pivot.columns):
    bars = ax.bar(x + j * width, pivot[col], width, label=col, color=colors[j], alpha=0.85)
    for bar, v in zip(bars, pivot[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('상품군')
ax.set_ylabel('MAPE (%)')
ax.set_title('베이스라인 모델 MAPE 비교 (Validation 기간)', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(pivot.index)
ax.axhline(y=12, color='red', linestyle='--', linewidth=1, label='목표 MAPE (12%)')
ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/10_baseline_mape.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: outputs/figures/10_baseline_mape.png')

저장: outputs/figures/10_baseline_mape.png


### 4.2 베이스라인 예측 vs 실제 시각화

Naive (7d) 모델의 예측값과 실제값을 시각적으로 비교한다.

In [14]:
fig, axes = plt.subplots(len(families), 1, figsize=(16, 3.5 * len(families)), sharex=True)

for i, family in enumerate(families):
    full_ts = full[full['family'] == family].set_index('date')['sales'].asfreq('D')
    full_ts = full_ts.interpolate(method='linear')
    
    val_sub = val[val['family'] == family].sort_values('date')
    val_dates = pd.DatetimeIndex(val_sub['date'])
    y_true = val_sub['sales'].values
    y_naive = naive_seasonal_forecast(full_ts, val_dates, lag=7)
    
    # 직전 30일 Train 데이터 포함
    train_tail = train[train['family'] == family].sort_values('date').tail(30)
    
    axes[i].plot(train_tail['date'], train_tail['sales'], color='gray', linewidth=1, label='Train (최근 30일)')
    axes[i].plot(val_dates, y_true, color='black', linewidth=1.2, label='실제 (Val)')
    axes[i].plot(val_dates, y_naive, color='#3498db', linewidth=1, linestyle='--', alpha=0.8, label='Naive 7d')
    axes[i].set_title(family, fontsize=11, fontweight='bold')
    axes[i].legend(loc='upper left', fontsize=8)

plt.suptitle('Naive (7d) 베이스라인: 예측 vs 실제', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/11_baseline_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: outputs/figures/11_baseline_forecast.png')

저장: outputs/figures/11_baseline_forecast.png


---
## 5. 요약 및 인사이트

### STL 분해 인사이트
- 모든 상품군에서 **주간 계절성(period=7)**이 뚜렷하게 관찰됨
- 주말/일요일에 매출이 높은 패턴이 공통적으로 나타남
- Trend 성분은 2013~2017 전반에 걸쳐 **상승 추세**를 보임
- 2016년 4월 지진 이후 일시적 매출 감소 → Residual에서 확인 가능

### 정상성 검정 인사이트
- 원본 시계열은 대부분 **비정상** (트렌드 + 계절성 존재)
- 1차 차분 후 대부분 **정상** → SARIMA에서 d=1 사용

### 베이스라인 성능
- **Naive (7d)**가 가장 단순하면서도 합리적인 베이스라인
- MA(30d)는 평활화 효과로 급격한 변동을 따라가지 못해 MAPE가 높음
- 목표 MAPE 12% 이하를 달성하려면 고급 모델이 필요함 확인

### Day 3 준비사항
- SARIMA: 주간 계절성(m=7), d=1 (또는 상품군별 검정 결과), 외생변수 포함
- 베이스라인 MAPE를 기준으로 개선율 측정

In [15]:
# 최종 요약 테이블
print('=' * 60)
print('Day 2 분석 요약')
print('=' * 60)
print()
print('1. STL 분해: 5개 상품군 모두 주간 계절성(period=7) 확인')
print()
print('2. 정상성 검정:')
for _, row in summary_df.iterrows():
    print(f"   {row['상품군']:15s} → 원본: {row['원본 결론']}, 차분: {row['차분 결론']}, 권장 d={row['권장 d']}")
print()
print('3. 베이스라인 MAPE (상품군별 최소):')
best_baseline = baseline_df.loc[baseline_df.groupby('family')['mape'].idxmin()]
for _, row in best_baseline.iterrows():
    print(f"   {row['family']:15s} → {row['model']} MAPE={row['mape']:.2f}%")
print()
overall_avg = best_baseline['mape'].mean()
print(f'   전체 평균 (최적 베이스라인): MAPE={overall_avg:.2f}%')
print(f'   목표 대비: {"달성" if overall_avg <= 12 else "미달성"} (목표: 12%)')
print()
print('4. 다음 단계: Day 3 (SARIMA), Day 4 (Prophet), Day 5 (XGBoost)')

Day 2 분석 요약

1. STL 분해: 5개 상품군 모두 주간 계절성(period=7) 확인

2. 정상성 검정:
   BEVERAGES       → 원본: 비정상, 차분: 정상, 권장 d=1
   CLEANING        → 원본: 불확실, 차분: 정상, 권장 d=1
   DAIRY           → 원본: 비정상, 차분: 정상, 권장 d=1
   GROCERY I       → 원본: 불확실, 차분: 정상, 권장 d=1
   PRODUCE         → 원본: 비정상, 차분: 정상, 권장 d=1

3. 베이스라인 MAPE (상품군별 최소):
   BEVERAGES       → Naive_7d MAPE=43.99%
   CLEANING        → MA_30d MAPE=120.03%
   DAIRY           → MA_30d MAPE=65.78%
   GROCERY I       → MA_7d MAPE=98.14%
   PRODUCE         → MA_30d MAPE=62.12%

   전체 평균 (최적 베이스라인): MAPE=78.01%
   목표 대비: 미달성 (목표: 12%)

4. 다음 단계: Day 3 (SARIMA), Day 4 (Prophet), Day 5 (XGBoost)
